# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hammadkhaliq-del/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


In [1]:
%pip install -q duckdb

import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('AccessToken')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # same dev partition used across Lane 2 weeks

DAILY_MONTH = f"{REL}/fact_content_daily_performance/month={MONTH}/data_0.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

probe = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{DAILY_MONTH}')").df()
print(f'{MONTH} partition row count:', probe['n'].iloc[0])
assert probe['n'].iloc[0] > 0, 'partition path returned zero rows -- fix before continuing.'
print('Ready.')


2026-03 partition row count: 9841378
Ready.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

**Feature vector for Lane 2 (Refresh / Content Opportunity Scoring).** One row = one `content_hash_id`, aggregated to the month (`{MONTH}`), same grain used in the Week 3 data contract and Week 5 model. Built from `fact_content_daily_performance` (GSC-available rows only) joined to the static `dim_content` attributes.

Engineered on top of the raw contract fields:
- `ctr_month` = `clicks_month / impressions_month` (guarded against divide-by-zero)
- `log_impressions_month`, `log_clicks_month` = `log1p()` of the raw sums, since both are heavy-tailed (a few pages carry most of the traffic) — same handling used in the Week 5/6 notebooks
- `position_bucket` = a categorical bucket of `avg_position_month` (top3 / page1 / page2 / page3+ / no-data), rather than treating position as linear — a jump from position 2→4 matters more than 40→42
- `days_since_last_optimized` = month-end date minus `dim_content.last_optimized_date`, filled with a large sentinel (999) when never optimized, plus a companion `never_optimized` flag so the model can tell "old but known" apart from "unknown" instead of silently treating both the same


In [2]:
feature_month_df = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS impressions_month,
           SUM(gsc_clicks) AS clicks_month,
           AVG(gsc_avg_position) AS avg_position_month
    FROM read_parquet('{DAILY_MONTH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

import numpy as np

# CTR, guarded against divide-by-zero (shouldn't occur given the HAVING filter, but explicit is safer than assumed)
feature_month_df['ctr_month'] = np.where(
    feature_month_df['impressions_month'] > 0,
    feature_month_df['clicks_month'] / feature_month_df['impressions_month'] * 100,
    0.0
)

# Heavy-tailed traffic columns -> log1p, per the auditing-signals / validating skills
feature_month_df['log_impressions_month'] = np.log1p(feature_month_df['impressions_month'])
feature_month_df['log_clicks_month'] = np.log1p(feature_month_df['clicks_month'])

def position_bucket(pos):
    if pos <= 0:
        return '0_no_position_data'
    elif pos <= 3:
        return '1_top3'
    elif pos <= 10:
        return '2_page1_4_10'
    elif pos <= 20:
        return '3_page2_11_20'
    elif pos <= 50:
        return '4_page3plus_21_50'
    else:
        return '5_deep_50plus'

feature_month_df['position_bucket'] = feature_month_df['avg_position_month'].apply(position_bucket)

# Join static content attributes
dim_content = con.sql(f"SELECT * FROM read_parquet('{DIM_CONTENT}')").df()
features = feature_month_df.merge(
    dim_content[['content_hash_id', 'client_hash_id', 'word_count', 'content_type',
                 'main_intent', 'last_optimized_date']],
    on=['content_hash_id', 'client_hash_id'], how='left'
)

month_end = pd.Timestamp('2026-03-31')
features['last_optimized_date'] = pd.to_datetime(features['last_optimized_date'])
features['never_optimized'] = features['last_optimized_date'].isna().astype(int)
features['days_since_last_optimized'] = (month_end - features['last_optimized_date']).dt.days
features['days_since_last_optimized'] = features['days_since_last_optimized'].fillna(999)

# content_type / main_intent are categorical -- explicit fill for missing rather than silent NaN passthrough
features['content_type'] = features['content_type'].fillna('unknown')
features['main_intent'] = features['main_intent'].fillna('unknown')

print('Feature frame shape:', features.shape)
features[['content_hash_id', 'impressions_month', 'ctr_month', 'position_bucket',
          'content_type', 'days_since_last_optimized', 'never_optimized']].head(5)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 15)


,content_hash_id,impressions_month,ctr_month,position_bucket,content_type,days_since_last_optimized,never_optimized
0,content_ac8663da7484669a,34.0,0.000000,2_page1_4_10,keyword article,999.0,1
1,content_39d7361b4945d504,77.0,0.000000,2_page1_4_10,keyword article,999.0,1
2,content_d49a012dcb924e31,329.0,0.000000,2_page1_4_10,keyword article,999.0,1
3,content_614baf2af4330bd7,772.0,0.129534,2_page1_4_10,keyword article,999.0,1
4,content_225dc9235023be5f,488.0,0.204918,3_page2_11_20,keyword article,999.0,1


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing values | Available before decision moment? |
|---|---|---|---|
| `impressions_month` / `clicks_month` | Raw GSC sums across the month, GSC-available rows only | Rows are filtered to `gsc_data_available IS TRUE` upstream; no NaNs reach here by construction | Yes — fully elapsed month-to-date sums, nothing looks past the report window |
| `ctr_month` | Click-through rate, `clicks/impressions*100` | Guarded with `np.where` against divide-by-zero, though the `HAVING impressions > 0` filter already prevents it | Yes — same window as impressions/clicks |
| `log_impressions_month` / `log_clicks_month` | `log1p()` of the raw sums, to tame the heavy right tail before any correlation or linear model sees it | Never NaN — `log1p` is defined everywhere the raw sum is (≥0) | Yes — derived from already-past sums |
| `position_bucket` | Categorical bucket of `avg_position_month` (top3 → deep/no-data) | `0_no_position_data` is itself a bucket, not a dropped row — a page with no measurable position is a real, informative state, not something to impute away | Yes — average of already-observed daily positions |
| `word_count`, `content_type`, `main_intent` | Static editorial attributes from `dim_content` | `content_type`/`main_intent` filled to `'unknown'` explicitly (not dropped, not silently left as NaN) so a model can learn "unknown" is its own category; `word_count` left as-is and checked for nulls below | Yes — set when the page was last edited, independent of any performance data |
| `days_since_last_optimized` | Month-end date minus `last_optimized_date` | Filled with sentinel `999` when `last_optimized_date` is null, **paired with** `never_optimized` flag — filling alone would make "never optimized" look identical to "optimized 999 days ago," which is a different and worse story | Yes — `last_optimized_date` is always in the past relative to the scoring date by construction |
| `never_optimized` | 1 if `last_optimized_date` is null, else 0 | N/A — this *is* the missingness flag | Yes |


In [3]:
# Verify the missingness claims made above, rather than asserting them
null_check = features[['impressions_month', 'clicks_month', 'ctr_month', 'word_count',
                        'content_type', 'main_intent', 'days_since_last_optimized']].isna().sum()
print('Null counts per column (should be 0 everywhere except possibly word_count):')
print(null_check)
print()
print('never_optimized rate:', round(features['never_optimized'].mean() * 100, 1), '% of rows')
print('position_bucket value counts:')
print(features['position_bucket'].value_counts())


Null counts per column (should be 0 everywhere except possibly word_count):
impressions_month                0
clicks_month                     0
ctr_month                        0
word_count                   55315
content_type                     0
main_intent                      0
days_since_last_optimized        0
dtype: int64

never_optimized rate: 77.5 % of rows
position_bucket value counts:
position_bucket
2_page1_4_10          81987
4_page3plus_21_50     33288
3_page2_11_20         32204
1_top3                16144
5_deep_50plus         11681
0_no_position_data     1434
Name: count, dtype: int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Running the taxonomy from `hunting-leakage-and-validating`: label-derived features, future/overlapping windows, decision-derived (product) flags.

**Suspect 1 — label-derived.** This week's frame has no label attached yet (that comes in Week 5), so there's no `is_declining_next` to derive a feature from directly. To make the hunt concrete, I deliberately manufacture a demo label the same way Week 3's data contract did — "is this page in the bottom half of `avg_position_month`?" — then add a rounded copy of `avg_position_month` itself as a feature, and show the score jump toward a suspicious near-1.0 number. Then I delete it.

**Suspect 2 — future/overlapping windows.** All features above are built from `month=2026-03` only. There is no 90-day trailing aggregate in this frame that could silently reach past the month boundary, so I verify that directly: the `MIN`/`MAX` of `report_date` feeding the aggregation is fully contained inside March 2026.

**Suspect 3 — decision-derived (product flags).** `dim_content` and `fact_content_daily_performance` carry no existing FlyRank priority score or flag in this slice (confirmed by checking the column list) — so there's nothing to accidentally use as a feature this week. Named explicitly here so it's on record, not just true by omission.


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# --- Suspect 2: confirm the window is fully contained in the stated month, no overlap ---
window_check = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date, COUNT(DISTINCT report_date) AS n_days
    FROM read_parquet('{DAILY_MONTH}')
""").df()
print('Window actually covered by this partition:')
print(window_check.to_string(index=False))
print('No 90-day or trailing aggregate is used in this frame, so there is no window that could reach past March.')
print()

# --- Suspect 1: manufacture a demo label, then attack it ---
df = features.dropna(subset=['impressions_month', 'avg_position_month', 'word_count']).copy()
df['needs_attention'] = (df['avg_position_month'] > df['avg_position_month'].median()).astype(int)

honest_cols = ['log_impressions_month', 'log_clicks_month', 'word_count']
X_honest = pd.get_dummies(df[honest_cols], drop_first=True)
y = df['needs_attention']

Xtr, Xte, ytr, yte = train_test_split(X_honest, y, test_size=0.3, random_state=42, stratify=y)
honest_auc = roc_auc_score(yte, LogisticRegression(max_iter=1000).fit(Xtr, ytr).predict_proba(Xte)[:, 1])
print(f'HONEST AUC (no leak): {honest_auc:.3f}')

# THE TRAP: add a rounded copy of the exact column the label was thresholded from
df['leaky_position_bucket'] = df['avg_position_month'].round(0)
leaky_cols = honest_cols + ['leaky_position_bucket']
X_leak = pd.get_dummies(df[leaky_cols], drop_first=True)

Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(X_leak, y, test_size=0.3, random_state=42, stratify=y)
leaky_auc = roc_auc_score(yte_l, LogisticRegression(max_iter=1000).fit(Xtr_l, ytr_l).predict_proba(Xte_l)[:, 1])
print(f'LEAKY AUC (with leaky_position_bucket): {leaky_auc:.3f}')
print(f'Jump: {honest_auc:.3f} -> {leaky_auc:.3f}')
print()
print('The jump toward ~1.0 confirms leaky_position_bucket is a near-restatement of needs_attention,')
print('not a real predictor -- the model is reading the label back, not learning anything about the page.')

# DELETE THE LEAK, KEEP THE HONEST NUMBER
del df['leaky_position_bucket']
print()
print(f'Honest AUC kept going forward: {honest_auc:.3f}')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Window actually covered by this partition:
  min_date   max_date  n_days
2026-03-01 2026-03-31      31
No 90-day or trailing aggregate is used in this frame, so there is no window that could reach past March.

HONEST AUC (no leak): 0.652
LEAKY AUC (with leaky_position_bucket): 0.998
Jump: 0.652 -> 0.998

The jump toward ~1.0 confirms leaky_position_bucket is a near-restatement of needs_attention,
not a real predictor -- the model is reading the label back, not learning anything about the page.

Honest AUC kept going forward: 0.652


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- `avg_position_month` (raw, unbucketed) as a direct feature alongside `position_bucket` — keeping both would double-count the same signal in two forms; `position_bucket` is the version that survives into the final frame.
- `client_hash_id`, `content_hash_id` — hash IDs, used only for joins and for the client-grouped split later, never handed to the model as inputs (same rule Week 3's data contract applied).
- Any GA4-sourced column (`ga4_*`, `sessions_*`, `ai_*`, `scroll_events`) — GA4 availability is well below 100% across clients (confirmed in the Week 3 data contract's availability query); including these now would silently bias the frame toward the subset of clients with GA4 access.
- Any existing FlyRank priority score or product flag — none exist in this month's slice for this table, but named explicitly per the leakage taxonomy's "decision-derived features" category: if one appeared later, it would be a baseline to beat, never an input.
- `is_deleted` / `is_published`-style status flags — not present in this slice, but if they were, they'd describe an outcome of editorial decisions, not a pre-decision signal, so they'd be excluded on the same "decision-derived" grounds.
- Raw `last_optimized_date` as a feature (kept only its derived `days_since_last_optimized` + `never_optimized`) — a raw date isn't directly usable by a model, and passing it in without the derived form would either need one-hot encoding hundreds of distinct dates (useless) or force an implicit, unstated assumption about how the model should read it.


In [5]:
# No extra computation needed for this section -- it's a classification/write-up,
# verified by the availability numbers already surfaced in the Week 3 data contract
# and the column-existence check below (confirms no product-flag columns are present to exclude).
cols_in_daily = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DAILY_MONTH}') LIMIT 1").df()
cols_in_content = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{DIM_CONTENT}') LIMIT 1").df()
flag_like = [c for c in list(cols_in_daily['column_name']) + list(cols_in_content['column_name'])
             if 'priority' in c.lower() or 'flag' in c.lower() or 'score' in c.lower()]
print('Columns matching priority/flag/score naming (should be empty this week):', flag_like)


Columns matching priority/flag/score naming (should be empty this week): []


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
